###Borrar y crear widgets

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "bakehouse_dev")

catalogName = dbutils.widgets.get("catalog")

# Bandera para mostrar prints en la fase de desarrollo
showPrint = False

### Insertar Gold - Customers

In [0]:
from pyspark.sql import functions as F

# Consultar tabla
df_silver = spark.table(f"{catalogName}.silver.customers_silver")

if showPrint:
  row_count, column_count = df_silver.count(), len(df_silver.columns)
  print(f"Row count: {row_count}")
  print(f"Column count: {column_count}")
  display(df_silver.limit(10))

Row count: 300
Column count: 13


customerID,first_name,last_name,gender,email_address,phone_number,address,continent,country,state,city,postal_zip_code,updatedate
2000000,Jimmy,Cross,false,ejacobson@example.net,(121)-983-071-62208,0203 Amber Cliffs Apt. 330,NORTH AMERICA,USA,null,Huntburgh,85109,2026-02-16T18:24:26.291Z
2000001,Beth,Park,false,juliaflores@example.com,(192)-874-999-01,307 White Roads,ASIA,Japan,Oregon,Jacksonhaven,74208,2026-02-16T18:24:26.291Z
2000002,Amber,Kirk,false,william99@example.net,(451)-667-662-4,4281 Teresa Island,NORTH AMERICA,USA,Texas,Scottland,30555,2026-02-16T18:24:26.291Z
2000003,Charles,Parsons,true,hillkyle@example.com,(936)-717-124-77716,102 Wilson Canyon Apt. 344,OCEANIA,Australia,Hawaii,Anthonyfurt,94248,2026-02-16T18:24:26.291Z
2000004,Charles,Gonzalez,true,jessica67@example.net,(928)-258-595-89952,6844 Richard Fords,NORTH AMERICA,USA,null,East Shannonland,45527,2026-02-16T18:24:26.291Z
2000005,Jared,Johnson,false,amy67@example.net,(690)-957-626-0289,13910 Ray Mountains Suite 603,OCEANIA,Australia,Kansas,New Willieland,17081,2026-02-16T18:24:26.291Z
2000006,Christopher,Miller,false,johnyates@example.net,(722)-867-886-33778,20110 Dalton Square Suite 177,ASIA,Japan,West Virginia,Tonyaville,24296,2026-02-16T18:24:26.291Z
2000007,Christopher,Garcia,true,zhangjohn@example.com,(266)-928-186-68854,825 Erika Shoal Suite 862,OCEANIA,Australia,Washington,Bakerside,16133,2026-02-16T18:24:26.291Z
2000008,Omar,Zamora,false,amandacooper@example.net,(001)-592-508-23204,18122 Sandra Valley Apt. 827,NORTH AMERICA,USA,New Jersey,East Julian,54414,2026-02-16T18:24:26.291Z
2000009,Jacob,Lane,false,richard55@example.net,(627)-978-935-25904,461 Sean Keys,OCEANIA,Australia,Hawaii,Lake Michael,48119,2026-02-16T18:24:26.291Z


In [0]:
# en el dataframe df_silver encontrar el numero de clientes por country y el numero de clientes por country, state. El resultado debe quedar en un solo dataframe

df_country = df_silver.groupBy("country").agg(F.count("customerID").alias("customer_count_country"))
df_country_state = df_silver.groupBy("country", "state").agg(F.count("customerID").alias("customer_count_country_state"))

df_result = df_country.join(df_country_state, on="country", how="outer")

# add metadata columns
df_result = df_result.withColumn("updatedate", F.current_timestamp())

if showPrint:
    display(df_result.orderBy("country", "state").limit(10))

country,customer_count_country,state,customer_count_country_state,updatedate
Australia,108,Alabama,1,2026-02-16T19:50:06.667Z
Australia,108,Alaska,1,2026-02-16T19:50:06.667Z
Australia,108,Arizona,5,2026-02-16T19:50:06.667Z
Australia,108,Arkansas,2,2026-02-16T19:50:06.667Z
Australia,108,California,2,2026-02-16T19:50:06.667Z
Australia,108,Colorado,1,2026-02-16T19:50:06.667Z
Australia,108,Connecticut,2,2026-02-16T19:50:06.667Z
Australia,108,Delaware,3,2026-02-16T19:50:06.667Z
Australia,108,Florida,1,2026-02-16T19:50:06.667Z
Australia,108,Georgia,2,2026-02-16T19:50:06.667Z


In [0]:
# Seleccionar y organizar columnas
df_gold = df_result.select("country", "customer_count_country", "state", "customer_count_country_state", "updatedate")

if showPrint:
    df_gold.printSchema()

root
 |-- country: string (nullable = true)
 |-- customer_count_country: long (nullable = true)
 |-- state: string (nullable = true)
 |-- customer_count_country_state: long (nullable = true)



In [0]:
# guardar los datos del dataframe en la tabla
df_gold.write.format("delta") \
    .mode("overwrite") \
    .options(mergeSchema="true") \
    .saveAsTable(f"{catalogName}.gold.customers_gold")